# Naija-Speech — EDA → Curate to Hugging Face

**What this notebook does**, end to end, on Google Colab:

1. **Stage 1 — EDA (explore):** look at the *catalogue* of the Nigerian speech data
   (how many clips, hours, accents, speakers) **without downloading any audio**. Cheap.
2. **Stage 2 — Curate (port to HF):** stream every data source, reshape each into **one
   shared schema**, and upload the result as Parquet shards to **your private Hugging Face
   dataset** — disk-safe, so Colab never fills up.
3. **Stage 3 — Preview:** load the curated dataset back and eyeball the first **20 rows**
   (`.head(20)`) to confirm the text and tags look right.

> You are new to ML — every stage has a short plain-English note first. Skip the notes if you
> already know the idea.

**You need:** a Hugging Face account + a **write** token. A W&B key is optional here (only used
in later training steps).

## 0 — Get the code and install dependencies

We clone the repo and `pip install`. The one non-obvious pin is **`datasets<4.0`** — AfriSpeech-200
ships a *loading script*, and Hugging Face removed script-loading in `datasets` 4.x. The
`requirements.txt` already pins this.

In [1]:
import os, subprocess

# Works on a fresh Colab (clones) or when already inside the repo.
if not os.path.exists("scripts/00_eda.py"):
    if not os.path.exists("naija-speech"):
        subprocess.run(
            ["git", "clone", "https://github.com/Johniblazee/naija-speech.git"],
            check=True,
        )
    os.chdir("naija-speech")

# Always pull latest so a RESUME session picks up pushed code changes.
subprocess.run(["git", "pull", "--ff-only"], check=False)
print("Working dir:", os.getcwd())

Working dir: /content/naija-speech


In [2]:
# Install everything the pipeline needs (datasets<4.0 pin lives in requirements.txt).
%pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 8.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.3 MB/s eta 0:00:0000:01


## 1 — Authenticate

- **Hugging Face token** (required): it uploads your curated dataset to *your* private HF storage,
  so it must have **write** access. Create one at
  <https://huggingface.co/settings/tokens>.
- **W&B key** (optional): only needed if you later log EDA/metrics to Weights & Biases. Press
  **Enter** to skip.

In [3]:
import os, getpass
from huggingface_hub import login

# Hugging Face — needs WRITE access (it uploads your dataset).
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (write): ")
login(os.environ["HF_TOKEN"])

# Weights & Biases — optional. Press Enter to skip.
if not os.environ.get("WANDB_API_KEY"):
    wb = getpass.getpass("W&B API key (optional, Enter to skip): ")
    if wb:
        os.environ["WANDB_API_KEY"] = wb

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 2 — Stage 1: EDA (metadata only, no audio)

**Plain English:** EDA = *Exploratory Data Analysis*. Before ordering every book, you read the
library **catalogue**. Here we read the transcript CSVs (durations, accents, speakers, domains)
and skip the audio entirely — so it runs in seconds and costs no disk.

**Why it matters for the thesis:** it quantifies **data imbalance** (e.g. Hausa is thin), which
justifies choices later like *weighted sampling* and *adding more sources*. These numbers feed
Chapter 4.

In [4]:
!python scripts/00_eda.py --config configs/data_afrispeech_ng.yaml --out outputs/eda

Nigerian clips: 6,126 | hours: 18.2 | speakers: 717 | accents: 80
Macro-accent split: {'Other': 4255, 'Yoruba': 997, 'Igbo': 562, 'Hausa': 312}
NG accents: ['afemai', 'afo', 'agatu', 'alago', 'anaang', 'angas', 'bagi', 'bajju', 'bassa', 'bassa-nge/nupe', 'bekwarra', 'benin', 'berom', 'bette', 'bini', 'brass', 'delta', 'ebiobo', 'ebira', 'edo', 'efik', 'eggon', 'ekene', 'eket', 'ekpeye', 'eleme', 'english', 'epie', 'esan', 'estako', 'etche', 'etsako', 'fulani', 'gbagyi', 'gerawa', 'hausa', 'hausa/fulani', 'ibani', 'ibibio', 'idah', 'idoma', 'igala', 'igarra', 'igbo', 'igbo and yoruba', 'ijaw', 'ijaw(nembe)', 'ika', 'ikulu', 'ikwere', 'ishan', 'isoko', 'itsekiri', 'izon', 'jaba', 'jukun', 'kalabari', 'kanuri', 'khana', 'kubi', 'mada', 'mwaghavul', 'nembe', 'ngas', 'nupe', 'nyandang', 'obolo', 'ogbia', 'ogoni', 'okirika', 'oklo', 'okrika', 'pidgin', 'tiv', 'ukwuani', 'urhobo', 'urobo', 'yala mbembe', 'yoruba', 'yoruba, hausa']
Report: outputs/eda/eda_report.md


In [5]:
# Show the generated report + charts inline.
from IPython.display import Markdown, Image, display
import glob, os

report = "outputs/eda/eda_report.md"
if os.path.exists(report):
    display(Markdown(open(report, encoding="utf-8").read()))

for png in sorted(glob.glob("outputs/eda/*.png")):
    display(Image(png))

# AfriSpeech-200 — Nigerian subset EDA

- Total Nigerian clips: **6,126**
- Total hours: **18.2**
- Unique speakers: **717**
- Distinct accents: **80**

## By macro_accent

| macro_accent   |   clips |   hours |   speakers |
|:---------------|--------:|--------:|-----------:|
| Other          |    4255 |    13.1 |        291 |
| Yoruba         |     997 |     2.7 |        228 |
| Igbo           |     562 |     1.5 |        126 |
| Hausa          |     312 |     0.9 |         72 |

## By accent

| accent          |   clips |   hours |   speakers |
|:----------------|--------:|--------:|-----------:|
| yoruba          |     997 |     2.7 |        228 |
| igbo            |     562 |     1.5 |        126 |
| hausa           |     312 |     0.9 |         72 |
| anaang          |     151 |     0.4 |          8 |
| english         |     150 |     0.7 |         11 |
| eggon           |     137 |     0.5 |          5 |
| ukwuani         |     133 |     0.4 |          7 |
| ijaw            |     128 |     0.3 |         23 |
| idoma           |     110 |     0.3 |         22 |
| mada            |     109 |     0.5 |          2 |
| bini            |     104 |     0.4 |          4 |
| berom           |     102 |     0.3 |          4 |
| ikwere          |      92 |     0.3 |          6 |
| yoruba, hausa   |      89 |     0.3 |          5 |
| ibibio          |      88 |     0.3 |         15 |
| kanuri          |      86 |     0.5 |          7 |
| itsekiri        |      82 |     0.2 |          3 |
| ekpeye          |      80 |     0.3 |          2 |
| mwaghavul       |      78 |     0.2 |          2 |
| bajju           |      72 |     0.2 |          2 |
| pidgin          |      71 |     0.2 |          8 |
| ekene           |      68 |     0.2 |          1 |
| urhobo          |      67 |     0.3 |         13 |
| ika             |      65 |     0.2 |          4 |
| jaba            |      65 |     0.2 |          2 |
| angas           |      65 |     0.2 |          1 |
| brass           |      62 |     0.3 |          2 |
| ikulu           |      61 |     0.1 |          1 |
| ebira           |      61 |     0.2 |         16 |
| eleme           |      60 |     0.3 |          2 |
| oklo            |      58 |     0.2 |          1 |
| fulani          |      56 |     0.1 |          7 |
| izon            |      55 |     0.2 |          7 |
| agatu           |      55 |     0.1 |          1 |
| epie            |      55 |     0.1 |          2 |
| ijaw(nembe)     |      54 |     0.1 |          2 |
| igarra          |      54 |     0.2 |          1 |
| okirika         |      54 |     0.2 |          1 |
| bekwarra        |      53 |     0.1 |          1 |
| alago           |      51 |     0.2 |          2 |
| ogbia           |      51 |     0.1 |          4 |
| gbagyi          |      51 |     0.2 |          4 |
| khana           |      50 |     0.1 |          2 |
| nupe            |      50 |     0.1 |          4 |
| etche           |      49 |     0.2 |          1 |
| delta           |      49 |     0.1 |          2 |
| bassa           |      49 |     0.2 |          1 |
| kubi            |      45 |     0.1 |          1 |
| tiv             |      45 |     0.1 |          6 |
| jukun           |      44 |     0.1 |          2 |
| urobo           |      43 |     0.2 |          3 |
| igbo and yoruba |      42 |     0.1 |          2 |
| ibani           |      42 |     0.1 |          1 |
| kalabari        |      41 |     0.1 |          5 |
| isoko           |      39 |     0.1 |          5 |
| obolo           |      37 |     0.1 |          1 |
| efik            |      37 |     0.1 |          4 |
| edo             |      36 |     0.1 |          3 |
| idah            |      34 |     0.1 |          1 |
| nembe           |      32 |     0.1 |          6 |
| bassa-nge/nupe  |      31 |     0.1 |          3 |
| yala mbembe     |      29 |     0.1 |          1 |
| eket            |      28 |     0.1 |          1 |
| esan            |      27 |     0.1 |          4 |
| afo             |      26 |     0   |          1 |
| ebiobo          |      25 |     0.1 |          1 |
| etsako          |      25 |     0.1 |          1 |
| nyandang        |      25 |     0.1 |          1 |
| ogoni           |      24 |     0.1 |          1 |
| ishan           |      23 |     0.1 |          1 |
| benin           |      21 |     0.1 |          1 |
| bagi            |      20 |     0.1 |          1 |
| estako          |      20 |     0.1 |          1 |
| ngas            |      19 |     0.1 |          1 |
| afemai          |      17 |     0   |          1 |
| gerawa          |      13 |     0.1 |          1 |
| igala           |      12 |     0   |          6 |
| hausa/fulani    |      10 |     0   |          1 |
| okrika          |      10 |     0   |          1 |
| bette           |       3 |     0   |          1 |

## By domain

| domain   |   clips |   hours |   speakers |
|:---------|--------:|--------:|-----------:|
| clinical |    3559 |    10.2 |        459 |
| general  |    2567 |     8   |        356 |

## By gender

| gender   |   clips |   hours |   speakers |
|:---------|--------:|--------:|-----------:|
| Male     |    3252 |    10.4 |        434 |
| Female   |    2779 |     7.5 |        448 |
| Other    |      95 |     0.3 |         19 |

## By age_group

| age_group   |   clips |   hours |   speakers |
|:------------|--------:|--------:|-----------:|
| 19-25       |    3476 |     9.4 |        397 |
| 26-40       |    1280 |     3.9 |        168 |
| 41-55       |    1092 |     4   |         87 |
| <18yrs      |     123 |     0.4 |         33 |
| 56yrs>      |      41 |     0.1 |          4 |

## By split

| split      |   clips |   hours |   speakers |
|:-----------|--------:|--------:|-----------:|
| test       |    4827 |    14.8 |        554 |
| validation |    1299 |     3.4 |        163 |

## Figures

![clips_by_macro_accent](figures/clips_by_macro_accent.png)
![clips_by_accent](figures/clips_by_accent.png)
![clips_by_domain](figures/clips_by_domain.png)
![clips_by_gender](figures/clips_by_gender.png)
![clips_by_split](figures/clips_by_split.png)
![duration_hist](figures/duration_hist.png)

## 3 — Stage 2: Curate all sources into ONE Hugging Face dataset

**The idea (plain English):** our data comes from **different datasets in different shapes**
(AfriSpeech-200 = short read clips; AfriSpeech-Dialog = long recorded conversations). We convert
every one into **a single shared table** so training can pull from all of them at once. Adding a
new dataset later = writing *one small adapter*, nothing else changes.

### The unified schema (the shared table's columns)

| Column | Meaning |
|---|---|
| `audio` | the waveform (kept at its **native** sample rate — see the resampling note below) |
| `text` / `text_raw` | normalized transcript / original transcript |
| `source` | which dataset it came from (`afrispeech-200`, `afrispeech-dialog`, …) |
| `language` | `en` / `en-codeswitch` / `pcm` (Nigerian Pidgin) / `yor` / `hau` / `ibo` |
| `task` | `stt` or `tts` |
| `accent`, `macro_accent` | raw accent, grouped into Yoruba / Igbo / Hausa / Other |
| `domain` | e.g. `clinical`, `conversational` |
| `speaker_id`, `gender`, `age_group` | speaker metadata |
| `duration` | seconds |
| `quality` | `unrated` / `clean` / `noisy` — used later to filter clean clips for **TTS** |
| `license` | so every clip carries its usage terms |

### Why streaming (and why your disk won't fill)

The old approach downloaded whole datasets **plus** a second local copy → Colab's disk filled and
nothing reached HF. Instead we **stream**: read a row → reshape it → buffer a few hundred → write
**one** Parquet shard → **upload it to HF** → **delete it locally**. Peak disk ≈ one shard.

### Note: audio resampling — *store native, resample late*

Whisper needs **16 kHz mono**, but we **store audio at its original rate** and only convert to
16 kHz **at training time**. Why: downsampling is **one-way and lossy** (it throws away everything
above 8 kHz). If we stored at 16 kHz, the later **TTS** track could never get the **24 kHz** it
wants for natural-sounding speech. So: keep the source faithful, resample late, per-task.

*(Full tradeoff table is in the chat / thesis §3. Short version: 16 kHz = model-compatible, 3×
smaller, keeps all speech intelligibility; the cost is discarded high frequencies that STT doesn't
need but TTS does.)*

### Note: the AfriSpeech-Dialog adapter

Its clips are long **multi-speaker conversations**; Whisper wants short single-speaker clips. The
adapter (`parse_dialog_turns`) reads the transcript's timestamps (`MM:SS:CC`) and `[Speaker N]:`
labels and **slices each conversation into per-turn utterances** — one short labelled clip per
speaker turn.

### Pick the run mode

The full Nigerian subset is large and **free Colab sessions time out / disconnect** — so the
pipeline **checkpoints every accent** and can resume across sessions.

- **First full run** → `FULL = True`, `RESUME = False`. Wipes the dataset and starts fresh,
  marking each accent done as it finishes.
- **After a disconnect** → `FULL = True`, `RESUME = True`, and re-run this section. It **skips
  accents already done** and continues — finished work is never re-downloaded. Repeat each new
  session until it completes.
- **Just validating?** → `FULL = False` runs a tiny smoke (~`MAX_PER_SPLIT` rows per source) in a
  minute.

> Expect the full port to span **several Colab sessions** — that's normal; resume makes it safe.
> To get the STT slice moving sooner, you can port only `[igbo, yoruba, hausa]` first (set
> `afrispeech_configs` in `configs/data_afrispeech_ng.yaml`), then expand later.

In [6]:
FULL   = True    # True = port the ENTIRE Nigerian subset; False = tiny smoke run
RESUME = False   # True = CONTINUE an interrupted FULL run (skip done accents, no wipe)
MAX_PER_SPLIT = 20   # only used when FULL is False

mode = ("FULL port" if FULL else "SMOKE") + (" + RESUME" if (FULL and RESUME) else "")
print("Mode:", mode)

Mode: FULL port


In [7]:
# --push streams every source into your private HF dataset (disk-safe: peak disk ~ one shard).
cmd = "python scripts/01_build_corpus.py --push"
if not FULL:
    cmd += f" --max-per-split {MAX_PER_SPLIT}"   # smoke
elif RESUME:
    cmd += " --resume"                            # continue where the last session stopped

print("$", cmd, "\n")
!{cmd}

$ python scripts/01_build_corpus.py --push 

README.md: 100% 18.3k/18.3k [00:00<00:00, 8.10MB/s]
afrispeech-200.py: 100% 11.6k/11.6k [00:00<00:00, 18.1MB/s]
accent_stats.py: 100% 30.4k/30.4k [00:00<00:00, 46.9MB/s]
[curate] Nigerian accents to stream (74): afemai, afo, agatu, alago, anaang, angas, bagi, bajju, bassa, bekwarra, benin, berom, bette, bini, brass, delta, ebiobo, ebira, edo, efik, eggon, ekene, eket, ekpeye, eleme, english, epie, esan, estako, etche, etsako, fulani, gbagyi, gerawa, hausa, ibani, ibibio, idah, idoma, igala, igarra, igbo, ijaw, ika, ikulu, ikwere, ishan, isoko, itsekiri, izon, jaba, jukun, kalabari, kanuri, khana, kubi, mada, mwaghavul, nembe, ngas, nupe, nyandang, obolo, ogbia, ogoni, okirika, oklo, okrika, pidgin, tiv, ukwuani, urhobo, urobo, yoruba
Reading metadata...: 125it [00:00, 657.37it/s]
Creating parquet from Arrow format: 100% 2/2 [00:11<00:00,  5.52s/ba]
[curate] train/afrispeech-200/afemai: 125 clips in 1 shard(s)
  [warn] afrispeech-200 afo/trai

**What to look for in the output above:**
- one line **per accent**, e.g. `[curate] train/afrispeech-200/igbo: N clips in K shard(s)`,
  plus `[curate] train/afrispeech-dialog/dialog: N clips …`
- on a **resume** run, finished accents show `[resume] skip done: train/afrispeech-200/igbo`
- uploaded shards: `uploaded data/train-afrispeech-200-igbo-00000.parquet (…)`
- the final check: **`verify audio decodes on 'train': OK`**

If you see `[warn] could not list afrispeech configs … using cfg['accents']`, the network hiccuped
and it fell back to just 3 accents — re-run to get the full set.

## 4 — Stage 3: Preview the curated data (`.head(20)`)

Load the dataset back **by streaming** (so we don't re-download audio just to look at text/tags)
and show the first 20 rows. Note: a Hugging Face `Dataset` has **no** `.head()` — that's a pandas
method — so we take 20 rows and view them as a pandas `DataFrame`.

In [8]:
from datasets import load_dataset, Audio
import pandas as pd

REPO = "Johniblazee/naija-speech-afrispeech-ng"  # = hf_curated_repo in configs/data_afrispeech_ng.yaml

# Stream + don't decode audio (decode=False) — we only want to eyeball text + tags cheaply.
stream = (load_dataset(REPO, split="train", streaming=True)
          .cast_column("audio", Audio(decode=False)))

rows = list(stream.take(20))
df = pd.DataFrame(rows).drop(columns=["audio"])   # drop the raw bytes from the preview
df.head(20)

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/75 [00:00<?, ?it/s]

,text,text_raw,source,language,task,accent,macro_accent,domain,speaker_id,gender,age_group,duration,quality,license
0,the patient had massive cocaine induced rhabdo...,The patient had massive Cocaine induced rhabdo...,afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,6.345986,unrated,CC-BY-NC-SA-4.0
1,radiopharmaceutical data 21 1 mci f 18 fdg sat...,RADIOPHARMACEUTICAL DATA: 21.1 mCi F-18 FDG S...,afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,30.264988,unrated,CC-BY-NC-SA-4.0
2,irritable bowel syndrome with constipation tab...,Irritable bowel syndrome with constipation. TA...,afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,17.452993,unrated,CC-BY-NC-SA-4.0
3,conus medullaris syndrome solution drops ophth...,"Conus medullaris syndrome. SOLUTION/DROPS, OPH...",afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,17.596985,unrated,CC-BY-NC-SA-4.0
4,laceration without foreign body of left great ...,Laceration without foreign body of left great ...,afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,23.294989,unrated,CC-BY-NC-SA-4.0
5,sequestration of lung tablet oral levetiraceta...,"Sequestration of lung. TABLET, ORAL LEVETIRACE...",afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,13.592993,unrated,CC-BY-NC-SA-4.0
6,extensive anterior st t wave changes which are...,Extensive anterior ST-T wave changes which are...,afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,11.382993,unrated,CC-BY-NC-SA-4.0
7,neonatology np notepeswaddled in open cribafof...,Neonatology NP NotePEswaddled in open cribAFOF...,afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,30.130999,unrated,CC-BY-NC-SA-4.0
8,she passed away at 2 10 pm on06 04 1987 with f...,She passed away at 2: 10 PM on06/04/1987 with ...,afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,12.503991,unrated,CC-BY-NC-SA-4.0
9,voiding and stooling lg amts,Voiding and stooling lg amts.,afrispeech-200,en,stt,afemai,Other,clinical,,Male,26-40,3.484989,unrated,CC-BY-NC-SA-4.0


### (Optional) Listen to one clip — at the 16 kHz STT training rate

This is the *only* place we actually decode + resample audio, to hear what the STT model will get.

In [9]:
from datasets import load_dataset, Audio
import IPython.display as ipd

ex = next(iter(
    load_dataset(REPO, split="train", streaming=True)
    .cast_column("audio", Audio(sampling_rate=16000))   # decode + resample to 16 kHz
))
print(f"[{ex['source']} | {ex['macro_accent']} | {ex['domain']}] {ex['text'][:80]}")
ipd.Audio(ex["audio"]["array"], rate=ex["audio"]["sampling_rate"])

Resolving data files:   0%|          | 0/100 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/75 [00:00<?, ?it/s]

[afrispeech-200 | Other | clinical] the patient had massive cocaine induced rhabdomyolysis


## What's next

- Add the **Common Voice (NG)** adapter (one more `*_stream` generator in `src/curate.py`).
- Then the **STT vertical slice**: zero-shot Whisper baseline → LoRA fine-tune (Unsloth) →
  accent-stratified WER → W&B. That produces the first real Chapter 4/5 numbers.

Progress is logged in `docs/PROGRESS.md`.